# LiveBench Generative Inference Notebook
This notebook is a complete interactive-readable script to help you get LiveBench LLM's inferenced texts easily.
It can also be easily adapted to get inferenced texts from other datasets like `GPQA-Diamond` and `MMLU-Pro` and output in the same LiveBench's style format.

**Requirement:**
* Need OPENROUTER_API_KEY, please put API into `os.environ["OPEN_ROUTER_API_KEY"]`

* Prepare problems in LiveBench's style: see example at `data/livebench_reasoning_problems.csv`

* Put input's full-path in `CSV_INPUT`

* you can use to inference only one or some of LiveBench's task categories (see `CATEGORY_LIST` below)
Often, some reasoning models some will take 9-12 hours for tasks like mathematics and reasoning, so we recommend inferencing one by one at first.

* modify `TEMPERATURE` and `MAX_TOKENS` up to your preferences.

In [ ]:
# --- CONFIGURATION ---
MODEL_TYPE = "openrouter"  # "local", "openai", "gemini", "openrouter"

MODEL_NAME = "openai/gpt-oss-20b" #openai/gpt-oss-20b:free will use prompt for training data (be aware)
BASE_URL = "https://openrouter.ai/api/v1" # for openrouter

CATEGORY_LIST = [
                # "language",
                "reasoning", 
                 # "instruction_following", 
                 "coding", 
                # "math" 
                ]
START_INDEX = 0
END_INDEX = 10000

TEMPERATURE = 1.0 # https://unsloth.ai/docs/models/gpt-oss-how-to-run-and-fine-tune#running-gpt-oss
MAX_TOKENS = 16384 # 24576, 32768 ## 24576 is max for 2.5-flash-lise
PROMPT_PREFIX = "" # optional

DEBUG=False
WAIT_API_TIME = 5 # wait 5 seconds, if there's some API error

CSV_INPUT = "data/livebench_reasoning_problems.csv"

In [3]:
import json
import time
import os
import torch
import pandas as pd
import numpy as np
import uuid
from typing import List, Dict, Any, Optional, Union, Tuple

# ==========================================
# 1. MODEL WRAPPERS
# ==========================================

class LiveBenchModel:
    """Base class to standardize generation across Local and API models."""
    def generate(self, messages: List[Dict[str, str]], temperature: float, max_tokens: int) -> Tuple[str, int]:
        raise NotImplementedError

class LocalHFModel(LiveBenchModel):
    """
    Wrapper for a local HuggingFace model using the high-level 'pipeline' abstraction.
    Supports device_map="auto", chat templates, and reasoning tokens.
    """
    def __init__(self, model_path: str):
        from transformers import pipeline
        
        print(f"Loading pipeline for: {model_path}...")
        self.model_name = model_path
        
        # trust_remote_code=True is often needed for newer architectures (e.g. Qwen/Phi)
        self.pipe = pipeline(
            "text-generation",
            model=model_path,
            device_map="auto",
            torch_dtype="auto",
            trust_remote_code=True
        )
        self.tokenizer = self.pipe.tokenizer

    def generate(self, messages: List[Dict[str, str]], temperature: float, max_tokens: int) -> Tuple[str, int]:
        
        # 1. Apply Chat Template
        try:
            # Try with thinking enabled (for reasoning models)
            prompt = self.tokenizer.apply_chat_template(
                messages, 
                tokenize=False, 
                add_generation_prompt=True,
                enable_thinking=True
            )
        except TypeError:
            # Fallback for standard models that don't support 'enable_thinking' arg
            prompt = self.tokenizer.apply_chat_template(
                messages, 
                tokenize=False, 
                add_generation_prompt=True
            )
        except Exception as e:
            # Ultimate fallback
            print(f"Warning: Chat template failed ({e}). using raw concatenation.")
            prompt = "\n".join([m['content'] for m in messages])

        # 2. Generate
        outputs = self.pipe(
            prompt,
            max_new_tokens=max_tokens,
            temperature=temperature,
            do_sample=True if temperature > 0 else False,
            return_full_text=False, # Return ONLY the new generated tokens
        )
        
        generated_text = outputs[0]['generated_text']
        
        # 3. Calculate Token Count
        # We tokenize the output text to get the count
        num_tokens = len(self.tokenizer(generated_text, add_special_tokens=False)['input_ids'])

        response = None # local model doesn't have full response like API models
        return generated_text, num_tokens, response

class OpenAIAPIModel(LiveBenchModel):
    """Wrapper for OpenAI-compatible APIs."""
    def __init__(self, api_key: str, model_name: str,
                 base_url: str = None,
                 extra_body: str = {} # extra_body can either be here or in generate()
                ):
        try:
            from openai import OpenAI
        except ImportError:
            raise ImportError("Please install openai: pip install openai")
        
        self.client = OpenAI(api_key=api_key, base_url=base_url)
        self.model_name = model_name
        self.extra_body = extra_body

    def generate(self, messages: List[Dict[str, str]], 
                 temperature: float, 
                 max_tokens: int,
                 ) -> Tuple[str, int]:
        response = self.client.chat.completions.create(
            model=self.model_name,
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens,
            extra_body=self.extra_body
        )
        
        # openrouter standard for reasoning-field
        reasoning = getattr(response.choices[0].message, 'reasoning', None)
        content = response.choices[0].message.content

        if reasoning:
            content = f"<think>\n{reasoning}\n</think>\n{content}"
            
        # Extract usage stats if available
        usage = getattr(response, 'usage', None)
        total_tokens = usage.completion_tokens if usage else 0

        return content, total_tokens, response

In [ ]:
# # Uncomment this cell if you want gemini instead of openrouter
# from google.genai import types
# class GeminiAPIModel(LiveBenchModel):
#     def __init__(self, api_key: str, model_name: str):        
#         self.client = genai.Client(api_key=api_key)
#         self.model_name = model_name
#         self.SAFETY_SETTINGS = [ # reduce API error from filter
#                                 types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="BLOCK_NONE"),
#                                 types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="BLOCK_NONE"),
#                                 types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="BLOCK_NONE"),
#                                 types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="BLOCK_NONE"),
#                                 types.SafetySetting(category="HARM_CATEGORY_CIVIC_INTEGRITY", threshold="BLOCK_NONE"),
#                                 ]

#     def generate(self, messages: List[Dict[str, str]], temperature: float, max_tokens: int,
#                 include_thoughts: bool=True) -> Tuple[str, int]:

#         response = self.client.models.generate_content(
#                           model=self.model_name,
#                           contents=messages,
#                           config=genai.types.GenerateContentConfig(
#                                   temperature=temperature,
#                                   safety_settings=self.SAFETY_SETTINGS,
#                                   thinking_config=genai.types.ThinkingConfig(
#                                   thinking_budget=max_tokens,
#                                   include_thoughts=include_thoughts
#                                  )
#                           )
#         )
        
#         content = response.text
#         # OR: response.usage_metadata.thoughts_token_count + response.usage_metadata.candidates_token_count
#         total_tokens = response.usage_metadata.total_token_count - response.usage_metadata.prompt_token_count
        
#         return content, total_tokens, response

In [5]:
def run_livebench_single_turn(
    model: LiveBenchModel,
    model_type:str,
    question_row: Dict[str, Any],
    prompt_prefix: str = "",
    temperature: float = 0.8,
    max_tokens: int = 32768,
    debug=False
) -> Optional[Dict[str, Any]]:
    """
    Performs inference for a single question row.
    Strictly handles SINGLE-TURN tasks (Math, Reasoning, Language).
    """
    
    # 1. EXTRACT PROMPT
    # print(question_row.get("turns", []))
    user_prompt_text = question_row['turns'][0]
    if not user_prompt_text:
        print(f"Warning: Empty/invalid prompt for question_id={question_row.get('question_id')}")
        return None
    
    final_user_prompt = user_prompt_text
    final_user_prompt = prompt_prefix + "\n" + user_prompt_text if prompt_prefix else user_prompt_text
    
    if model_type == "gemini":
        # https://googleapis.github.io/python-genai/
        messages = final_user_prompt #[{"role": "user", "parts": [final_user_prompt]}]
    else:
        messages = [{"role": "user", "content": final_user_prompt}]

    # 2. GENERATE
    try_again = False
    response = None
    try:
        output_text, total_tokens, response = model.generate(messages, temperature, max_tokens)
    except:
        try_again = True

    if output_text=="":
        try:
            print(f'>>> get NULL output_text, while reasoning is :{response.choices[0].message.reasoning}')
        except:
            pass
        try_again = True
    
    if model_type == "gemini" and response is not None and response.candidates[0].finish_reason.value == 'RECITATION':
        try_again = True
    # TODO: add other condition, like finish_reason.value == 'SAFETY'
    
    if try_again:
        time.sleep(WAIT_API_TIME)
        try: # e.g. TIMEOUT or RECITATION
            print("\\n *** TRY AGAIN !!*** \\n")
            if model_type == "gemini" and response is not None:
                print("FINISH REASON:", response.candidates[0].finish_reason.value)
            response = None
            output_text, total_tokens, response = model.generate(messages, temperature=temperature*1.5, max_tokens=max_tokens/2)
        except:
            print("\\n *** STILL COULDN'T GET RESULTS !!*** \\n")
            output_text, total_tokens = "ERROR", 0

    # 3. FORMAT OUTPUT
    category = question_row.get('category', 'unknown')
    task = question_row.get('task', 'unknown')
    subtask = question_row.get("subtask", task)
    answer_id = str(uuid.uuid4())

    result = {
        "question_id": question_row.get("question_id", "unknown"),
        "turns":  question_row.get("turns", [""]),
        "answer_id": answer_id,
        "ground_truth": question_row.get("ground_truth", ""),
        "model_id": getattr(model, "model_name", "custom_experiment_model"),
        "choices": [{
            "index": 0, 
            "turns": [output_text] 
        }],
        "tstamp": time.time(),
        "category": category,
        "task": task,
        "subtask": subtask,
        "total_output_tokens": total_tokens,
        # for spatial, zebra
        "livebench_release_date": question_row.get("livebench_release_date", "1-1-2024"), # default is old date
        
        # these 4 items are for coding
        'public_test_cases': question_row.get('public_test_cases', ""),
        'private_test_cases': question_row.get('private_test_cases', ""),
        'original_json': question_row.get('original_json', ""),
        "partial_solution": question_row.get("partial_solution", ""),
        
        "full_response" : response
    }
    
    return result

In [ ]:
%%time

print(f"{MODEL_TYPE} - {MODEL_NAME}")
if MODEL_TYPE == "local":
    runner_model = LocalHFModel(MODEL_NAME)
elif MODEL_TYPE == "openai":
    api_key = os.environ.get("OPENAI_API_KEY", "")
    if not api_key: raise RuntimeError("Missing OPENAI_API_KEY")
    runner_model = OpenAIAPIModel(api_key=api_key, model_name=MODEL_NAME)
elif MODEL_TYPE == "openrouter":
    api_key = os.environ.get("OPEN_ROUTER_API_KEY", "")
    if not api_key: raise RuntimeError("Missing OPEN_ROUTER_API_KEY")

    extra_body = {}
    if MAX_TOKENS > 0:
        extra_body = {
                "include_reasoning": True 
            }
    runner_model = OpenAIAPIModel(api_key=api_key, model_name=MODEL_NAME, base_url=BASE_URL, extra_body=extra_body)
else:
    from google import genai
    api_key = os.environ.get("GOOGLE_API_KEY", "")
    if not api_key: raise RuntimeError("Missing GOOGLE_API_KEY")
    runner_model = GeminiAPIModel(api_key=api_key, model_name=MODEL_NAME)


results_list: List[Dict[str, Any]] = []
for cat_name in CATEGORY_LIST:
    
    # from datasets import load_dataset
    # print(f"Loading Dataset: livebench/{cat_name} ...")
    # ds = load_dataset(f"livebench/{cat_name}", split='test')
    # df_cat = ds.to_pandas()
    if cat_name == "instruction_following":
        cat_name = "if"

    
    df_cat = pd.read_csv(CSV_INPUT)
    import ast # IMPORTANT: fix csv str format of list/dict
    df_cat['turns'] = df_cat['turns'].apply(ast.literal_eval)
    
    # 3. Process Questions
    df_subset = df_cat.iloc[START_INDEX:END_INDEX]
    print(f"\nStarting Experiment on {len(df_subset)} tasks using {runner_model.model_name}...\n" + "="*40)
    
    for _, row in df_subset.iterrows():
        answer = run_livebench_single_turn(
            model=runner_model,
            model_type=MODEL_TYPE,
            question_row=row.to_dict(),
            prompt_prefix=PROMPT_PREFIX,
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS,
            debug=DEBUG
        )

        if answer:
            results_list.append(answer)
            # Preview
            try:
                print(f"ID: {answer['question_id']} | Task: {answer['task']}")
                print(f"question: {answer['turns'][0][:1000]}")
                print(f"Tokens: {answer['total_output_tokens']}")
                print(f"Output Preview: {answer['choices'][0]['turns'][0][:200]}...")
                print(f"Ending Preview: {answer['choices'][0]['turns'][0][-200:]}")
                print(f"ground_truth ending preview: {answer['ground_truth'][-200:]}")
                print("-" * 40)
            except Exception as e:
                print(f"Error: {e}")




openrouter - openai/gpt-oss-20b

Starting Experiment on 128 tasks using openai/gpt-oss-20b...
ID: 009f4f7e8275c73de2eae601d76ff7163f613880788d870c1484b1c2a1383fbd | Task: coding_completion
question: ### Instructions: You are an expert Python programmer. You will be given a question (problem specification) and the first lines of Python solution to this problem, and will write in Python the remaining lines of the program to produce a correct Python program that matches the specification and passes all tests. You will NOT return anything except for the second part of the program that you wrote.
### Question:
You are given a 0-indexed string s and an integer k.
You are to perform the following partitioning operations until s is empty:

Choose the longest prefix of s containing at most k distinct characters.
Delete the prefix from s and increase the number of partitions by one. The remaining characters (if any) in s maintain their initial order.

Before the operations, you are allowed to ch

In [ ]:



final_output = pd.DataFrame(results_list)
print(f"\nFinal Output (DataFrame):\n")

output_filename = "experiment_results_livebench.csv"
final_output.to_csv(output_filename, index=False) # dict/list structure will lost here, so also save as jsonl
print(f"Saved DataFrame to {output_filename}")


Final Output (DataFrame):

Saved DataFrame to experiment_results_livebench.csv


In [8]:
final_output.head(3)

,question_id,turns,answer_id,ground_truth,model_id,choices,tstamp,category,task,subtask,total_output_tokens,livebench_release_date,public_test_cases,private_test_cases,original_json,partial_solution,full_response
0,009f4f7e8275c73de2eae601d76ff7163f613880788d87...,[### Instructions: You are an expert Python pr...,106423f6-e145-4fb9-90fd-c0b94a809701,,openai/gpt-oss-20b,"[{'index': 0, 'turns': ['<think> We need to fi...",1.768660e+09,coding,coding_completion,coding_completion,14162,2024-06-24,"[{""input"": ""\""accca\""\n2"", ""output"": ""3"", ""tes...",eJyknMmuLstVhBnwINYdM3CDBeJJkPAk+75Z2WciJF6C9y...,{'question_title': 'maximize-the-number-of-par...,class Solution(object):\n def maxPartitions...,ChatCompletion(id='gen-1768659532-rdz1QG6qVc2Y...
1,0119bdbf9e79f05e988b6d3408e2fdc0b7e4bc11e81ff9...,[### Instructions: You are an expert Python pr...,3449aa44-7843-4c55-a485-b924ef8e3f96,,openai/gpt-oss-20b,"[{'index': 0, 'turns': ['<think> We need to wr...",1.768660e+09,coding,coding_completion,coding_completion,427,2024-07-26,"[{""input"": ""[2, 11, 10, 1, 3]\n10"", ""output"": ...",eJztls1qFEEQxz148imGOS9SX10f4s2XEGIOIhECsglk9i...,{'question_title': 'minimum-operations-to-exce...,class Solution(object):\n def minOperations...,ChatCompletion(id='gen-1768659668-pmigy5a5Yguv...
2,01c73e7f5bd7b486915f97829994aca9b95d6116c510d9...,[### Instructions: You are an expert Python pr...,6bf700dd-f347-4df5-98b7-4e401bdf225b,,openai/gpt-oss-20b,"[{'index': 0, 'turns': ['<think> We need to fi...",1.768660e+09,coding,LCB_generation,LCB_generation,285,2024-06-24,"[{""input"": ""[1, 2, 3, 4, 5]"", ""output"": ""7"", ""...",eJzlVE1LAzEQ9SDevPkDwp6DJJlsPvwlQulBpEJB2kK3Bx...,{'question_title': 'maximum-strong-pair-xor-i'...,NaN,ChatCompletion(id='gen-1768659670-jJzFO8pilpnv...


In [9]:
final_output.tail(3)

,question_id,turns,answer_id,ground_truth,model_id,choices,tstamp,category,task,subtask,total_output_tokens,livebench_release_date,public_test_cases,private_test_cases,original_json,partial_solution,full_response
125,f78df88e6755702752e1c216f9734cbf1803a006546503...,[### Instructions: You are an expert Python pr...,ed1f3753-5901-4123-bfe3-2aa0805e7177,,openai/gpt-oss-20b,"[{'index': 0, 'turns': ['<think> We need to wr...",1.768661e+09,coding,LCB_generation,LCB_generation,345,2024-06-24,"[{""input"": ""1 2 3 4 5 6 7 8 9\n4 5 6 7 8 9 1 2...",eJzNVrtuFEEQJCDiK0YXW0jznuEjiEEsGRc4WVu6c2AhJP...,"{'question_title': 'Number Place', 'question_c...",NaN,ChatCompletion(id='gen-1768661326-c64B9pL34IG9...
126,fe6312c9e8cb798517e3695c245cbbfbf4fd6dc7379036...,[### Instructions: You are an expert Python pr...,c9343eb2-e870-4c57-88d6-dfce1e0ddecc,,openai/gpt-oss-20b,"[{'index': 0, 'turns': ['<think> We need to fi...",1.768661e+09,coding,coding_completion,coding_completion,3906,2024-06-24,"[{""input"": ""[1, 5, 3, 9, 8]\n2"", ""output"": ""[1...",eJyU3d2qdEuWFmAPPPUeijoT6iAjxl+kVyK0Hoi00CDdgt...,{'question_title': 'make-lexicographically-sma...,class Solution(object):\n def lexicographic...,ChatCompletion(id='gen-1768661331-mmUAhHrIczem...
127,ffdb71c563e5f8e936ab2649a069bfbdbebcde991c6ae1...,[### Instructions: You are an expert Python pr...,2713c62e-a242-48c2-9132-c3d57dc2bb28,,openai/gpt-oss-20b,"[{'index': 0, 'turns': ['<think> We need to de...",1.768661e+09,coding,LCB_generation,LCB_generation,1798,2024-07-26,"[{""input"": ""[8, 4, 2, 30, 15]"", ""output"": ""tru...",eJyll02KHEcQhb3wAXyEZtaNib/MjPRJDEILYSQQCElYPQ...,{'question_title': 'find-if-array-can-be-sorte...,NaN,ChatCompletion(id='gen-1768661362-qwQEXB4Y7BZb...


In [10]:
!ls -sh

total 248M
247M experiment_results_livebench.csv  308K __notebook__.ipynb


In [11]:
final_output.iloc[2].choices

[{'index': 0,
  'turns': ["<think>\nWe need to find maximum XOR among pairs (x,y) from array that satisfy |x - y| <= min(x,y). We can brute force all pairs including same indices. nums length up to 50, so 2500 pairs - trivial.\n\nSo implement double loop including same elements. Condition: abs(x - y) <= min(x, y). Compute xor and track max. Return max.\n\nEdge cases: no pairs? at least one element, so min >=1. At least one strong pair? For a single element, pair with itself satisfies |x-x|=0 <= min(x,x)=x. So we always have at least that pair.\n\nOk.\n\nImplementation straightforward.\n\nTime O(n^2). We'll import List.\n\nLet's produce full code.\n</think>\n```python\nfrom typing import List\n\nclass Solution:\n    def maximumStrongPairXor(self, nums: List[int]) -> int:\n        max_xor = 0\n        n = len(nums)\n        for i in range(n):\n            for j in range(i, n):  # include same element and avoid duplicate computations\n                x, y = nums[i], nums[j]\n             